# 7. Takvim etkileri: efsane mi, gerçek mi?

Yaygın iddialar: "Pazartesi kötüdür", "ay başında para girer", "tatil öncesi yükselir", "opsiyon vadesi haftası oynaktır". Nasdaq 100'ün uzun geçmişiyle test ediyoruz.

**Yöntem:** Her grup diğer tüm günlerle Welch t-testiyle karşılaştırılır. 20 test yapıldığı için p değerleri **Benjamini–Hochberg** ile düzeltilir. Ayrıca etki verinin iki yarısında aynı yönde mi diye bakılır (kararlılık).

In [1]:
import sys, sqlite3
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from mi import db, stats as st, events as ev, charts
%matplotlib inline
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
con = db.connect()
prices = pd.read_sql('SELECT * FROM prices', con)
close, volume = st.wide(prices), st.wide(prices, 'volume')
r = st.returns(close)
assets = pd.read_sql('SELECT * FROM assets', con).set_index('code')
print(f'{close.shape[1]} varlık, {close.index[0]:%Y-%m-%d} – {close.index[-1]:%Y-%m-%d}, {len(prices):,} fiyat satırı')

109 varlık, 2021-11-01 – 2026-10-30, 130,652 fiyat satırı


In [2]:
from mi import calendar as cal
from mi.config import LONG_DIR
L = pd.read_csv(LONG_DIR / 'NDX.csv', parse_dates=['date']).set_index('date')['close']
C = cal.analyze(L)
print(f"{C['start']} – {C['end']}, {C['n_days']} gün; düzeltmesiz anlamlı {C['naive_significant']}, düzeltme sonrası {C['adj_significant']}")
T = pd.DataFrame(C['tests']).set_index('name')
T[['n', 'mean', 'rest', 'p', 'p_adj', 'diff_1', 'diff_2', 'verdict']].sort_values('p').round(4)

2000-01-04 – 2026-10-30, 6747 gün; düzeltmesiz anlamlı 2, düzeltme sonrası 0


,n,mean,rest,p,p_adj,diff_1,diff_2,verdict
name,,,,,,,,
gün:Cuma,1353,-0.04,0.06,0.04,0.49,-0.14,-0.07,yalnızca düzeltmesiz anlamlı
ay:Eylül,547,-0.08,0.06,0.05,0.49,-0.16,-0.13,yalnızca düzeltmesiz anlamlı
ay:Şubat,518,-0.03,0.05,0.24,0.90,-0.11,-0.06,fark yok
ay:Ekim,597,0.11,0.04,0.36,0.90,0.16,-0.01,fark yok
gün:Perşembe,1359,0.08,0.04,0.37,0.90,0.16,-0.07,fark yok
Ay dönümü (son gün + ilk 3 gün),1288,0.08,0.04,0.42,0.90,0.09,-0.00,fark yok
ay:Kasım,531,0.10,0.04,0.43,0.90,0.01,0.11,fark yok
ay:Aralık,547,-0.01,0.05,0.43,0.90,-0.02,-0.09,fark yok
gün:Çarşamba,1385,0.08,0.04,0.44,0.90,0.01,0.07,fark yok


## Neden düzeltme gerekiyor? Simülasyon
Tamamen rastgele getirilerle aynı 20 testi 500 kez yapalım: düzeltme olmadan kaç "anlamlı" sonuç çıkıyor?

In [3]:
rng = np.random.default_rng(0)
idx = L.index[L.index >= '2000-01-01']
F = cal.flags(idx[1:])
naive, adj = [], []
for _ in range(500):
    r = pd.Series(rng.normal(0, .015, len(F)), index=F.index)
    ps = [stats.ttest_ind(r[m], r[~m], equal_var=False).pvalue for m in
          [F.weekday == i for i in range(5)] + [F.month == m for m in range(1, 13)] + [F.tom, F.opex, F.pre_holiday]]
    naive.append(sum(p < .05 for p in ps)); adj.append(sum(cal.bh_adjust(np.array(ps)) < .05))
print(f'Rastgele veride ortalama düzeltmesiz "anlamlı" sonuç: {np.mean(naive):.2f}, en az bir tane çıkma olasılığı %{np.mean(np.array(naive) > 0) * 100:.0f}')
print(f'Düzeltmeden sonra: {np.mean(adj):.2f}, en az bir tane: %{np.mean(np.array(adj) > 0) * 100:.0f}')

Rastgele veride ortalama düzeltmesiz "anlamlı" sonuç: 0.96, en az bir tane çıkma olasılığı %62
Düzeltmeden sonra: 0.05, en az bir tane: %5


**Yorum:** Rastgele veride bile 20 testten yaklaşık biri "anlamlı" çıkar ve en az bir tane çıkma olasılığı yüksektir. Takvim etkisi iddialarının çoğu bu yanılgıdan doğar. Düzeltilmiş p değeri ve iki dönemde tutarlılık, gerçek bir etkiyi şanstan ayırmanın asgari şartıdır.